# Kiki SLM — Data Curation Notebook

**Goal:** Curate a high-quality 10K SFT dataset from 8 source datasets.

**Pipeline:**
1. Setup & install deps
2. Download & convert all 8 datasets to ChatML
3. Inspect raw data — see examples from each source
4. Exact dedup (MD5 hash)
5. Semantic dedup (sentence-transformer @ 0.85 threshold)
6. Length filtering (20-1500 tokens)
7. Intent balancing (no single intent > 25%)
8. Inject empty `<think></think>` blocks (Qwen3 non-thinking mode)
9. Stratified train/eval split
10. Final inspection & save

**Each step has detailed inspection outputs so you can see exactly what's happening to your data.**

## 1. Setup

In [ ]:
%%capture
# Install deps (Colab)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

!uv pip install --system datasets sentence-transformers pandas tabulate

In [ ]:
# Mount Google Drive first (needed before repo clone into Drive)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone / update repo
import os

REPO_DIR = "/content/drive/MyDrive/kiki-slm/repo"

if os.path.exists(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/dev
    print(f"Updated repo at {REPO_DIR}")
else:
    !git clone -b dev https://github.com/BlueSpringsAI/kiki-slm.git {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")

%cd {REPO_DIR}

# Install kiki package in dev mode so converters are available
!uv pip install --system -e ".[dev]" 2>&1 | tail -3

# ============================================================
# CONFIGURATION — tweak these
# ============================================================
DRIVE_DATA = "/content/drive/MyDrive/kiki-slm/data"
os.makedirs(DRIVE_DATA, exist_ok=True)

TARGET = 10_000          # Total curated examples
EVAL_RATIO = 0.10        # 10% eval split
SEMANTIC_THRESHOLD = 0.85 # Cosine similarity for near-dedup
MAX_INTENT_RATIO = 0.25   # No single intent > 25%
SEED = 42

print(f"\nTarget: {TARGET:,} examples")
print(f"Eval ratio: {EVAL_RATIO}")
print(f"Semantic dedup threshold: {SEMANTIC_THRESHOLD}")
print(f"Max intent ratio: {MAX_INTENT_RATIO}")

In [ ]:
import hashlib
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import display, HTML
from kiki.data.processors import ChatMLConverter

random.seed(SEED)

# Dataset config — 3 tiers
SFT_DATASETS = {
    # --- Tier 1: CS-core (60%) ---
    "bitext_cs": {
        "hf_id": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
        "converter": "bitext", "tier": 1, "tier_weight": 0.35,
        "description": "General CS (27K)",
    },
    "bitext_ecom": {
        "hf_id": "bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset",
        "converter": "bitext", "tier": 1, "tier_weight": 0.35,
        "description": "E-commerce (50K+)",
    },
    "customer_support_tickets": {
        "hf_id": "Tobi-Bueck/customer-support-tickets",
        "converter": "ticket", "tier": 1, "tier_weight": 0.30,
        "filter_language": "en", "description": "Helpdesk tickets (61.8K, English subset)",
    },
    # --- Tier 2: Tool/function calling (20%) ---
    "xlam_60k": {
        "hf_id": "Salesforce/xlam-function-calling-60k",
        "converter": "xlam", "tier": 2, "tier_weight": 0.55,
        "description": "Function calling (60K)",
    },
    "hermes_fc": {
        "hf_id": "NousResearch/hermes-function-calling-v1",
        "converter": "hermes", "tier": 2, "tier_weight": 0.45,
        "description": "Tool calling (11.6K)",
    },
    # --- Tier 3: Secondary domains (20%) ---
    "bitext_banking": {
        "hf_id": "bitext/Bitext-retail-banking-llm-chatbot-training-dataset",
        "converter": "bitext", "tier": 3, "tier_weight": 0.30,
        "description": "Banking (37K+)",
    },
    "bitext_insurance": {
        "hf_id": "bitext/Bitext-insurance-llm-chatbot-training-dataset",
        "converter": "bitext", "tier": 3, "tier_weight": 0.30,
        "description": "Insurance (38K+)",
    },
    "banking77": {
        "hf_id": "legacy-datasets/banking77",
        "converter": "banking77", "tier": 3, "tier_weight": 0.40,
        "description": "Banking intent 77-class (13K)",
    },
}

TIER_ALLOCATIONS = {1: 0.60, 2: 0.20, 3: 0.20}
TIER_NAMES = {1: "CS-core", 2: "Tool-calling", 3: "Secondary"}

# Show the plan
rows = []
for name, cfg in SFT_DATASETS.items():
    tier = cfg["tier"]
    budget = int(TARGET * TIER_ALLOCATIONS[tier] * cfg["tier_weight"] /
                 sum(v["tier_weight"] for v in SFT_DATASETS.values() if v["tier"] == tier))
    rows.append({"Dataset": name, "Tier": f"T{tier} ({TIER_NAMES[tier]})",
                 "Description": cfg["description"], "Target": budget,
                 "Download (w/ 20% buffer)": int(budget * 1.2)})

plan_df = pd.DataFrame(rows)
print("Dataset plan:")
display(plan_df)
print(f"\nDropped: arcee_agent (generic agent, low CS relevance), clinc_oos (general intent)")
print(f"Total target: {plan_df['Target'].sum():,} examples")

## 2. Download & Convert All Datasets

In [ ]:
def download_and_convert(name, config, num_samples, token=None):
    """Download a dataset, convert to ChatML, subsample."""
    hf_id = config["hf_id"]
    converter = ChatMLConverter.get_converter(config["converter"])

    print(f"\n{'─'*60}")
    print(f"  Downloading: {name} ({hf_id})")

    kwargs = {"path": hf_id, "split": "train"}
    if config.get("subset"):
        kwargs["name"] = config["subset"]

    t0 = time.monotonic()
    try:
        ds = load_dataset(**kwargs)
    except Exception as exc:
        print(f"  ERROR: Failed to download '{name}': {exc}")
        return []
    print(f"  Downloaded {len(ds):,} rows in {time.monotonic()-t0:.1f}s")

    # Language filter
    if config.get("filter_language") and "language" in ds.column_names:
        before = len(ds)
        ds = ds.filter(lambda x: x.get("language") == config["filter_language"])
        print(f"  Language filter ({config['filter_language']}): {before:,} -> {len(ds):,}")

    # Convert
    converted = []
    skipped = 0
    skip_reasons = Counter()

    for example in ds:
        try:
            result = converter(dict(example))
            messages = result.get("messages", [])
            if len(messages) < 2:
                skipped += 1; skip_reasons["too_few_messages"] += 1; continue
            has_assistant = any(
                m.get("role") == "assistant" and m.get("content", "").strip()
                for m in messages
            )
            if not has_assistant:
                skipped += 1; skip_reasons["no_assistant_content"] += 1; continue
            converted.append({"messages": messages, "source": name})
        except Exception:
            skipped += 1; skip_reasons["conversion_error"] += 1

    print(f"  Converted: {len(converted):,} | Skipped: {skipped} {dict(skip_reasons) if skip_reasons else ''}")

    # Subsample
    if num_samples and len(converted) > num_samples:
        converted = random.sample(converted, num_samples)
        print(f"  Subsampled to: {num_samples:,}")
    elif num_samples and len(converted) < num_samples:
        print(f"  WARNING: Only {len(converted):,} available (target: {num_samples:,})")

    return converted

# Download all datasets
all_examples = []
download_stats = []

for tier, tier_ratio in TIER_ALLOCATIONS.items():
    tier_budget = int(TARGET * tier_ratio)
    tier_datasets = {k: v for k, v in SFT_DATASETS.items() if v["tier"] == tier}
    total_tier_weight = sum(v["tier_weight"] for v in tier_datasets.values())

    print(f"\n{'='*60}")
    print(f"  TIER {tier}: {TIER_NAMES[tier]} (budget: {tier_budget:,})")
    print(f"{'='*60}")

    for name, config in tier_datasets.items():
        raw_target = int(tier_budget * config["tier_weight"] / total_tier_weight)
        download_target = int(raw_target * 1.2)

        examples = download_and_convert(name, config, download_target)
        all_examples.extend(examples)
        download_stats.append({
            "source": name, "tier": tier,
            "target": raw_target, "downloaded": len(examples),
        })

print(f"\n{'='*60}")
print(f"  TOTAL DOWNLOADED: {len(all_examples):,}")
print(f"{'='*60}")

if not all_examples:
    raise RuntimeError("No examples downloaded! Check network and HuggingFace access.")

## 3. Inspect Raw Data

Look at the source distribution, sample examples from each dataset, token length distribution, and intent breakdown — all **before** any filtering.

In [ ]:
# --- 3a. Source distribution table ---
source_counts = Counter(ex["source"] for ex in all_examples)
rows = []
for source, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    tier = SFT_DATASETS.get(source, {}).get("tier", "?")
    rows.append({
        "Source": source,
        "Tier": f"T{tier} ({TIER_NAMES.get(tier, '?')})",
        "Count": count,
        "%": f"{count/len(all_examples)*100:.1f}%",
    })

print(f"Raw dataset: {len(all_examples):,} total examples\n")
display(pd.DataFrame(rows))

In [ ]:
# --- 3b. Sample 2 examples from each source ---
# Shows the actual ChatML messages so you can verify format quality

by_source = {}
for ex in all_examples:
    by_source.setdefault(ex["source"], []).append(ex)

for source in sorted(by_source.keys()):
    examples = by_source[source]
    samples = random.sample(examples, min(2, len(examples)))
    print(f"\n{'='*70}")
    print(f"  SOURCE: {source} ({len(examples):,} examples)")
    print(f"{'='*70}")

    for i, ex in enumerate(samples):
        print(f"\n  --- Example {i+1} ---")
        for m in ex["messages"]:
            role = m["role"]
            content = m["content"]
            # Truncate long content for display
            preview = content[:300] + "..." if len(content) > 300 else content
            print(f"  [{role}]: {preview}")
        print()

In [ ]:
# --- 3c. Token length distribution (whitespace tokens) ---
# Note: whitespace tokens undercount BPE tokens by ~40%. A 1500 whitespace-token
# limit is roughly ~1000 BPE tokens — safely under max_seq_length=2048.
import matplotlib.pyplot as plt
import numpy as np

def get_token_length(ex):
    return sum(len(m.get("content", "").split()) for m in ex["messages"])

lengths = [get_token_length(ex) for ex in all_examples]
lengths_by_source = {}
for ex in all_examples:
    s = ex["source"]
    lengths_by_source.setdefault(s, []).append(get_token_length(ex))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall distribution
axes[0].hist(lengths, bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(x=20, color='red', linestyle='--', label='min=20')
axes[0].axvline(x=1500, color='red', linestyle='--', label='max=1500')
axes[0].set_title(f'Token Length Distribution (all {len(lengths):,} examples)')
axes[0].set_xlabel('Whitespace tokens')
axes[0].set_ylabel('Count')
axes[0].legend()

# Per-source box plot
source_names = sorted(lengths_by_source.keys())
box_data = [lengths_by_source[s] for s in source_names]
axes[1].boxplot(box_data, labels=source_names, vert=True)
axes[1].set_title('Token Length by Source')
axes[1].set_ylabel('Whitespace tokens')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Stats table
stats_rows = []
for source in source_names:
    lens = lengths_by_source[source]
    stats_rows.append({
        "Source": source, "Count": len(lens),
        "Min": min(lens), "Median": int(np.median(lens)),
        "Mean": int(np.mean(lens)), "Max": max(lens),
        "< 20 tokens": sum(1 for l in lens if l < 20),
        "> 1500 tokens": sum(1 for l in lens if l > 1500),
    })
display(pd.DataFrame(stats_rows))

In [ ]:
# --- 3d. Intent distribution (before filtering) ---

def extract_intent(ex):
    """Extract intent from assistant JSON response."""
    for m in ex.get("messages", []):
        if m.get("role") == "assistant":
            content = m.get("content", "")
            try:
                parsed = json.loads(content)
                return parsed.get("intent")
            except (json.JSONDecodeError, TypeError):
                match = re.search(r'"intent"\s*:\s*"([^"]+)"', content)
                if match:
                    return match.group(1)
    return None

intent_counts = Counter()
no_intent_count = 0
intent_by_source = {}

for ex in all_examples:
    intent = extract_intent(ex)
    if intent:
        intent_counts[intent] += 1
        intent_by_source.setdefault(ex["source"], Counter())[intent] += 1
    else:
        no_intent_count += 1

# Overall intent distribution
print(f"Intent distribution ({sum(intent_counts.values()):,} with intent, {no_intent_count:,} without):\n")
intent_rows = []
for intent, count in sorted(intent_counts.items(), key=lambda x: -x[1]):
    intent_rows.append({
        "Intent": intent,
        "Count": count,
        "%": f"{count/len(all_examples)*100:.1f}%",
        "Over 25%?": "YES" if count/len(all_examples) > MAX_INTENT_RATIO else "",
    })
display(pd.DataFrame(intent_rows))

# Intent heatmap by source
print(f"\nIntent x Source heatmap:")
all_intents = sorted(intent_counts.keys())
heatmap_rows = []
for source in sorted(intent_by_source.keys()):
    row = {"Source": source}
    for intent in all_intents:
        row[intent] = intent_by_source[source].get(intent, 0)
    heatmap_rows.append(row)
heatmap_df = pd.DataFrame(heatmap_rows).set_index("Source")
display(heatmap_df.style.background_gradient(cmap="YlOrRd", axis=None))

## 4. Exact Dedup

Remove examples with identical user+assistant content (MD5 hash, ignoring system prompt).

In [ ]:
before_exact = len(all_examples)

seen = set()
unique = []
duplicates_by_source = Counter()

for ex in all_examples:
    key_parts = [
        f"{m.get('role')}:{m.get('content', '')}"
        for m in ex.get("messages", [])
        if m.get("role") != "system"
    ]
    h = hashlib.md5("|".join(key_parts).encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique.append(ex)
    else:
        duplicates_by_source[ex["source"]] += 1

all_examples = unique
after_exact = len(all_examples)

print(f"Exact dedup: {before_exact:,} -> {after_exact:,} (removed {before_exact - after_exact:,})\n")

if duplicates_by_source:
    print("Duplicates removed per source:")
    dup_rows = [{"Source": s, "Duplicates Removed": c}
                for s, c in sorted(duplicates_by_source.items(), key=lambda x: -x[1])]
    display(pd.DataFrame(dup_rows))
else:
    print("No exact duplicates found.")

## 5. Semantic Dedup

Remove near-duplicates using sentence-transformer embeddings. Two examples with cosine similarity >= 0.85 on their user messages are considered duplicates.

This catches cases like:
- "I want a refund for my shoes" vs "I want a refund for my jacket" (same pattern, different noun)
- "Where is my order?" vs "Where's my order?" (trivial rephrasing)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

before_semantic = len(all_examples)
# Keep reference to pre-dedup list for per-source counting
pre_dedup_examples = all_examples

print(f"Computing embeddings for {len(all_examples):,} examples...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract user message text
texts = []
for ex in all_examples:
    user_msgs = [m.get("content", "") for m in ex["messages"] if m.get("role") == "user"]
    texts.append(" ".join(user_msgs) if user_msgs else "")

embeddings = st_model.encode(texts, batch_size=512, show_progress_bar=True, normalize_embeddings=True)
print(f"Embeddings shape: {embeddings.shape}")

# Sliding window dedup
# Note: compares against last 500 kept items only — duplicates further apart may be missed.
# For full pairwise dedup, install faiss-cpu and use the FAISS path in prepare_sft_curated.py.
keep_indices = []
removed_indices = []
removed_pairs = []  # Store some examples of what got removed
window_size = 500

for i in range(len(embeddings)):
    is_dup = False
    best_sim = 0.0
    best_j = -1
    for j in keep_indices[-window_size:]:
        sim = float(np.dot(embeddings[i], embeddings[j]))
        if sim >= SEMANTIC_THRESHOLD:
            is_dup = True
            best_sim = sim
            best_j = j
            break
    if not is_dup:
        keep_indices.append(i)
    else:
        removed_indices.append(i)
        if len(removed_pairs) < 10:  # Save first 10 for inspection
            removed_pairs.append((i, best_j, best_sim))

all_examples = [pre_dedup_examples[i] for i in keep_indices]
after_semantic = len(all_examples)

print(f"\nSemantic dedup: {before_semantic:,} -> {after_semantic:,} (removed {before_semantic - after_semantic:,})")

# Per-source breakdown of removed near-duplicates
removed_sources = Counter(pre_dedup_examples[idx]["source"] for idx in removed_indices)
if removed_sources:
    print(f"\nNear-duplicates removed per source:")
    sem_dup_rows = [{"Source": s, "Removed": c}
                    for s, c in sorted(removed_sources.items(), key=lambda x: -x[1])]
    display(pd.DataFrame(sem_dup_rows))

In [ ]:
# --- 5b. Inspect what got removed --- 
# Show pairs of near-duplicates so you can verify the threshold is right

if removed_pairs:
    print(f"Sample near-duplicate pairs (showing {len(removed_pairs)} of {len(removed_indices):,}):\n")
    for removed_idx, kept_idx, sim in removed_pairs:
        print(f"  Similarity: {sim:.4f}")
        print(f"  KEPT    [{kept_idx}]: {texts[kept_idx][:120]}...")
        print(f"  REMOVED [{removed_idx}]: {texts[removed_idx][:120]}...")
        print()
else:
    print("No near-duplicates found at threshold", SEMANTIC_THRESHOLD)

# Similarity distribution of removed pairs
if removed_indices:
    print(f"\nTip: If too many are being removed, raise SEMANTIC_THRESHOLD (currently {SEMANTIC_THRESHOLD})")
    print(f"     If not enough, lower it. Try 0.80 for aggressive, 0.90 for conservative.")

## 6. Length Filtering

Remove examples that are too short (< 20 tokens — likely garbage) or too long (> 1500 tokens — will get truncated at `max_seq_length=2048` after tokenization overhead).

In [ ]:
MIN_TOKENS = 20
MAX_TOKENS = 1500

before_length = len(all_examples)
filtered = []
removed_short = []
removed_long = []

for ex in all_examples:
    total_text = " ".join(m.get("content", "") for m in ex["messages"])
    token_count = len(total_text.split())
    if token_count < MIN_TOKENS:
        removed_short.append((ex, token_count))
    elif token_count > MAX_TOKENS:
        removed_long.append((ex, token_count))
    else:
        filtered.append(ex)

all_examples = filtered
after_length = len(all_examples)

print(f"Length filter: {before_length:,} -> {after_length:,}")
print(f"  Removed {len(removed_short)} too short (< {MIN_TOKENS} tokens)")
print(f"  Removed {len(removed_long)} too long (> {MAX_TOKENS} tokens)")

# Show examples of removed short
if removed_short:
    print(f"\nSample SHORT examples removed:")
    for ex, length in removed_short[:5]:
        user_msg = next((m["content"] for m in ex["messages"] if m["role"] == "user"), "")
        print(f"  [{length} tokens] [{ex['source']}] {user_msg[:100]}")

# Show examples of removed long
if removed_long:
    print(f"\nSample LONG examples removed:")
    for ex, length in removed_long[:5]:
        user_msg = next((m["content"] for m in ex["messages"] if m["role"] == "user"), "")
        print(f"  [{length} tokens] [{ex['source']}] {user_msg[:100]}")

## 7. Intent Balancing

Cap any single intent at 25% of the dataset. If `billing_inquiry` has 40% of examples, we downsample it to 25% so the model doesn't over-index on one intent.

In [ ]:
before_balance = len(all_examples)

# Classify examples
with_intent = []
without_intent = []
for i, ex in enumerate(all_examples):
    intent = extract_intent(ex)
    if intent:
        with_intent.append((i, intent))
    else:
        without_intent.append(i)

counts_before = Counter(intent for _, intent in with_intent)
max_per_intent = int(len(all_examples) * MAX_INTENT_RATIO)

over_represented = {intent: count for intent, count in counts_before.items() if count > max_per_intent}

if over_represented:
    print(f"Over-represented intents (max allowed: {max_per_intent:,} = {MAX_INTENT_RATIO*100:.0f}%):\n")
    for intent, count in sorted(over_represented.items(), key=lambda x: -x[1]):
        print(f"  {intent:<25s} {count:>5,} ({count/len(all_examples)*100:.1f}%) -> will be capped to {max_per_intent:,}")

    # Downsample
    intent_indices = {}
    for idx, intent in with_intent:
        intent_indices.setdefault(intent, []).append(idx)

    rng = random.Random(42)
    keep_indices = set(without_intent)
    for intent, indices in intent_indices.items():
        if intent in over_represented:
            rng.shuffle(indices)
            keep_indices.update(indices[:max_per_intent])
        else:
            keep_indices.update(indices)

    all_examples = [all_examples[i] for i in sorted(keep_indices)]
    after_balance = len(all_examples)

    print(f"\nIntent balance: {before_balance:,} -> {after_balance:,} (removed {before_balance - after_balance:,})")

    # Show after distribution
    counts_after = Counter()
    for ex in all_examples:
        intent = extract_intent(ex)
        if intent:
            counts_after[intent] += 1

    print(f"\nIntent distribution AFTER balancing:")
    balance_rows = []
    for intent in sorted(set(list(counts_before.keys()) + list(counts_after.keys()))):
        before = counts_before.get(intent, 0)
        after = counts_after.get(intent, 0)
        balance_rows.append({
            "Intent": intent,
            "Before": before,
            "After": after,
            "Removed": before - after,
            "% After": f"{after/len(all_examples)*100:.1f}%",
        })
    display(pd.DataFrame(balance_rows))
else:
    after_balance = before_balance
    print(f"All intents are within the {MAX_INTENT_RATIO*100:.0f}% threshold. No balancing needed.")

## 8. Final Cap to Target Size

If we still have more than the target after all filtering, proportionally downsample each source to hit the target while preserving the distribution.

In [ ]:
before_cap = len(all_examples)

if len(all_examples) > TARGET:
    by_source = {}
    for ex in all_examples:
        by_source.setdefault(ex["source"], []).append(ex)

    ratio = TARGET / len(all_examples)
    capped = []
    cap_details = []

    for source, source_examples in sorted(by_source.items()):
        n = max(1, int(len(source_examples) * ratio))
        random.shuffle(source_examples)
        capped.extend(source_examples[:n])
        cap_details.append({
            "Source": source,
            "Before": len(source_examples),
            "After": n,
            "Removed": len(source_examples) - n,
        })

    all_examples = capped
    print(f"Capped to target: {before_cap:,} -> {len(all_examples):,}\n")
    display(pd.DataFrame(cap_details))
else:
    print(f"Already at or below target ({len(all_examples):,} <= {TARGET:,}). No capping needed.")

## 9. Inject Empty `<think></think>` Blocks

Per Qwen3 team recommendation ([GitHub #1429](https://github.com/QwenLM/Qwen3/discussions/1429)): add empty think blocks to all assistant messages to explicitly train non-thinking mode. This prevents the model from generating reasoning tokens at inference time.

In [ ]:
modified = 0
for ex in all_examples:
    for m in ex["messages"]:
        if m.get("role") == "assistant":
            content = m.get("content", "")
            if "<think>" in content:
                continue
            m["content"] = "<think>\n\n</think>\n\n" + content
            modified += 1

print(f"Injected empty <think> blocks into {modified:,} assistant messages")

# Show a before/after example
sample = random.choice(all_examples)
print(f"\nSample assistant message (first 400 chars):")
for m in sample["messages"]:
    if m["role"] == "assistant":
        print(m["content"][:400])
        break

## 10. Stratified Train/Eval Split

Split 90/10 stratified by source so every source is represented in the eval set.

In [ ]:
rng = random.Random(SEED)

by_source = {}
for ex in all_examples:
    by_source.setdefault(ex["source"], []).append(ex)

train_data = []
eval_data = []
split_details = []

for source, source_examples in sorted(by_source.items()):
    rng.shuffle(source_examples)
    eval_size = max(1, int(len(source_examples) * EVAL_RATIO))
    eval_data.extend(source_examples[:eval_size])
    train_data.extend(source_examples[eval_size:])
    split_details.append({
        "Source": source,
        "Total": len(source_examples),
        "Train": len(source_examples) - eval_size,
        "Eval": eval_size,
    })

rng.shuffle(train_data)
rng.shuffle(eval_data)

print(f"Stratified split: {len(train_data):,} train + {len(eval_data):,} eval\n")
display(pd.DataFrame(split_details))

## 11. Final Inspection

Full summary of the curated dataset before saving — source distribution, intent distribution, token length stats, and sample examples from train and eval sets.

In [ ]:
# --- 11a. Pipeline summary ---
print(f"{'='*70}")
print(f"  CURATION PIPELINE SUMMARY")
print(f"{'='*70}")
print(f"  Downloaded:          {before_exact:>8,}")
print(f"  After exact dedup:   {after_exact:>8,}  (-{before_exact - after_exact:,})")
print(f"  After semantic dedup:{after_semantic:>8,}  (-{after_exact - after_semantic:,})")
print(f"  After length filter: {after_length:>8,}  (-{after_semantic - after_length:,})")
print(f"  After intent balance:{after_balance:>8,}  (-{before_balance - after_balance:,})")
print(f"  After final cap:     {len(all_examples):>8,}")
print(f"  Train set:           {len(train_data):>8,}")
print(f"  Eval set:            {len(eval_data):>8,}")
print(f"{'='*70}")

In [ ]:
# --- 11b. Final source distribution (side-by-side train vs eval) ---
train_sources = Counter(ex["source"] for ex in train_data)
eval_sources = Counter(ex["source"] for ex in eval_data)

all_sources = sorted(set(list(train_sources.keys()) + list(eval_sources.keys())))
final_rows = []
for source in all_sources:
    t = train_sources.get(source, 0)
    e = eval_sources.get(source, 0)
    tier = SFT_DATASETS.get(source, {}).get("tier", "?")
    final_rows.append({
        "Source": source,
        "Tier": f"T{tier}",
        "Train": t,
        "Train %": f"{t/len(train_data)*100:.1f}%",
        "Eval": e,
        "Eval %": f"{e/len(eval_data)*100:.1f}%",
    })

print("Final source distribution:")
display(pd.DataFrame(final_rows))

In [ ]:
# --- 11c. Final intent distribution ---
final_intents = Counter()
for ex in all_examples:
    intent = extract_intent(ex)
    if intent:
        final_intents[intent] += 1

fig, ax = plt.subplots(figsize=(12, 5))
intents = [k for k, _ in sorted(final_intents.items(), key=lambda x: -x[1])]
counts = [final_intents[k] for k in intents]
colors = ['#e74c3c' if c/len(all_examples) > MAX_INTENT_RATIO else '#3498db' for c in counts]
bars = ax.bar(intents, counts, color=colors, edgecolor='black', alpha=0.8)
ax.axhline(y=len(all_examples)*MAX_INTENT_RATIO, color='red', linestyle='--',
           label=f'{MAX_INTENT_RATIO*100:.0f}% cap ({int(len(all_examples)*MAX_INTENT_RATIO)})')
ax.set_title(f'Final Intent Distribution ({len(all_examples):,} examples)')
ax.set_ylabel('Count')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# No-intent examples (tool-calling datasets)
no_intent = sum(1 for ex in all_examples if extract_intent(ex) is None)
print(f"\nExamples without intent (tool-calling format): {no_intent:,} ({no_intent/len(all_examples)*100:.1f}%)")

In [ ]:
# --- 11d. Sample 3 random TRAIN examples (full messages) ---
print(f"{'='*70}")
print(f"  SAMPLE TRAIN EXAMPLES")
print(f"{'='*70}")

for i, ex in enumerate(random.sample(train_data, min(3, len(train_data)))):
    print(f"\n--- Train Example {i+1} [source: {ex['source']}] ---")
    for m in ex["messages"]:
        role = m["role"]
        content = m["content"]
        if role == "system":
            print(f"  [system]: {content[:80]}...")
        else:
            preview = content[:500] + "..." if len(content) > 500 else content
            print(f"  [{role}]: {preview}")
    print()

# Sample 2 EVAL examples
print(f"\n{'='*70}")
print(f"  SAMPLE EVAL EXAMPLES")
print(f"{'='*70}")

for i, ex in enumerate(random.sample(eval_data, min(2, len(eval_data)))):
    print(f"\n--- Eval Example {i+1} [source: {ex['source']}] ---")
    for m in ex["messages"]:
        role = m["role"]
        content = m["content"]
        if role == "system":
            print(f"  [system]: {content[:80]}...")
        else:
            preview = content[:500] + "..." if len(content) > 500 else content
            print(f"  [{role}]: {preview}")
    print()

## 12. Save & Upload

Save to Google Drive as JSONL files ready for training.

In [ ]:
train_path = f"{DRIVE_DATA}/kiki_sft_curated_train.jsonl"
eval_path = f"{DRIVE_DATA}/kiki_sft_curated_eval.jsonl"

for path, data, label in [(train_path, train_data, "Train"), (eval_path, eval_data, "Eval")]:
    with open(path, "w") as f:
        for ex in data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"{label}: {path}")
    print(f"  {len(data):,} examples, {size_mb:.1f} MB")

print(f"\nFiles saved to Google Drive. Ready for training!")
print(f"\nNext step — run training notebook with:")
print(f"  TRAIN_FILE = \"{train_path}\"")
print(f"  EVAL_FILE  = \"{eval_path}\"")
print(f"  --epochs 2 --lora-r 16 --lora-alpha 32")